# VLA 연구 · 10회차 테스트
## MuJoCo 정리 시나리오 데이터 제작 & SmolVLM SFT 데이터셋 만들기

* 정리가 완료된 상태의 씬과,  정리가 필요한 상태의 씬을 정의하고 생성한다 (MuJoCo로 편집)
* 그 과정 씬들을 생성하고, 과정에 설명을 붙인다 (예 : 오른쪽으로 밀기, 들어서 왼쪽으로 옮기기 등)
* 이때엔 초기에 구상한  pick, push, reorder, ignore은 우선 생각하지 말고, 시나리오에 가장 적합한 과정 설명이 무엇인지 고안하고, 이후 이를 분류한다.
* 그 후 최종 라벨링과 상세 설명을 확정짓고, 파인튜닝을 위한 데이터셋을 만든다.
> 산출물: MuJoCo XML, 렌더링 이미지, 과정 라벨 JSON/JSONL, TRL SFT용 `messages` 데이터셋, Google Drive 저장본


[참고 자료]
* https://github.com/markusgrotz/mujoco-scene-editor
* https://huggingface.co/learn/cookbook/fine_tuning_smol_vlm_sft_trl
* https://huggingface.co/learn/cookbook/ko/ko_fine_tuning_vlm_dpo_smolvlm_instruct
+ 다음 시간 (VLA 연동) 활용 자료 : https://github.com/jeongeun980906/lerobot-mujoco-tutorial

---
## 0. 실행 환경과 Google Drive 준비

권장 환경은 Colab GPU입니다. MuJoCo 렌더링만 할 때는 CPU도 가능하지만, 이후 SmolVLM 파인튜닝까지 이어가려면 GPU 런타임을 사용하세요.

In [ ]:
# 0-1. 패키지 설치 — 처음 1회 실행 후 Runtime > Restart session 권장

import os
import sys
import subprocess

os.environ.setdefault('MUJOCO_GL', 'egl')
RUN_INSTALL = True

if sys.version_info >= (3, 13):
    print('⚠️ mujoco-scene-editor는 Python 3.10~3.12 환경을 권장합니다. Colab 또는 Python 3.12 가상환경을 사용하세요.')

PACKAGES = [
    'mujoco>=3.3,<4.0',
    'mujoco-scene-editor>=0.1.3',
    'datasets>=3.0.0',
    'transformers>=4.49.0',
    'trl>=0.22.0',
    'peft>=0.17.0',
    'bitsandbytes>=0.47.0',
    'accelerate',
    'Pillow>=10.4.0,<12.0',
    'matplotlib',
    'ipywidgets',
]

if RUN_INSTALL:
    cmd = [sys.executable, '-m', 'pip', 'install', '-q', '--upgrade'] + PACKAGES
    subprocess.check_call(cmd)
    print('✓ 설치 완료')
    print('⚠️ 설치 후 Runtime > Restart session 으로 재시작하고 0-2부터 실행하세요.')
else:
    print('RUN_INSTALL=False: 설치를 건너뜁니다.')

In [ ]:
# 0-2. 경로, Drive, 기본 환경 확인

from pathlib import Path
import json
import os
import platform
import shutil
import sys
from collections import Counter

IN_COLAB = 'google.colab' in sys.modules or Path('/content').exists()
WORK_DIR = Path('/content/session10_process_dataset') if IN_COLAB else Path.cwd() / 'session10_process_dataset'
DATA_DIR = WORK_DIR / 'data'
XML_DIR = DATA_DIR / 'xml'
IMAGE_DIR = DATA_DIR / 'images'
LABEL_DIR = DATA_DIR / 'labels'
EDITED_XML_DIR = DATA_DIR / 'edited_xml'
DATASET_DIR = DATA_DIR / 'sft_dataset'
REPORT_DIR = WORK_DIR / 'reports'

for path in [WORK_DIR, DATA_DIR, XML_DIR, IMAGE_DIR, LABEL_DIR, EDITED_XML_DIR, DATASET_DIR, REPORT_DIR]:
    path.mkdir(parents=True, exist_ok=True)

TEAM_FOLDER = 'VLA_Team_2026'
TEAM_ROOT = None

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    TEAM_ROOT = Path('/content/drive/MyDrive') / TEAM_FOLDER
    if TEAM_ROOT.exists():
        for name in ['data', 'images', 'models', 'results', 'notebooks']:
            (TEAM_ROOT / name).mkdir(exist_ok=True)
        print('✓ 팀 공유 폴더:', TEAM_ROOT)
    else:
        print('⚠️ 팀 공유 폴더가 보이지 않습니다:', TEAM_ROOT)
else:
    print('Colab이 아니므로 Drive 마운트를 건너뜁니다.')

print('Python  :', sys.version.split()[0])
print('Platform:', platform.platform())
print('WORK_DIR:', WORK_DIR)

In [ ]:
# 0-3. 기본 라이브러리 임포트

import math
import random
import xml.etree.ElementTree as ET

import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
from IPython.display import display

try:
    import mujoco
    print('✓ mujoco:', mujoco.__version__)
except Exception as e:
    mujoco = None
    print('⚠️ mujoco import 실패:', repr(e))

---
## 1. 정리 시나리오 정의

오늘의 시나리오는 테이블 위 물체 정리입니다.

* 빨간 상자: 왼쪽 보관 영역으로 옮겨야 하는 목표 물체
* 파란 블록: 빨간 상자 앞을 막고 있어 오른쪽으로 밀어야 하는 장애물
* 노란 원통: 뒤쪽 정리 영역으로 옮겨야 하는 물체
* 회색 블록: 오른쪽 정리 영역으로 옮겨야 하는 물체

여기서는 처음부터 `pick/push/reorder/ignore`로 분류하지 않습니다. 각 장면에서 실제로 자연스러운 **다음 과정 설명**을 먼저 적습니다.

In [ ]:
# 1-1. 정리 전/중/후 상태 정의

TABLE_TOP_Z = 0.425

OBJECT_SPECS = {
    'target_red_box': {'geom_type': 'box', 'size': (0.045, 0.045, 0.040), 'rgba': '0.95 0.10 0.10 1.0', 'role': 'target_to_left_zone'},
    'blue_blocker': {'geom_type': 'box', 'size': (0.060, 0.035, 0.035), 'rgba': '0.10 0.25 0.95 1.0', 'role': 'obstacle_to_right_side'},
    'yellow_cylinder': {'geom_type': 'cylinder', 'size': (0.035, 0.035, 0.040), 'rgba': '0.95 0.75 0.10 1.0', 'role': 'object_to_back_zone'},
    'gray_box': {'geom_type': 'box', 'size': (0.040, 0.040, 0.030), 'rgba': '0.45 0.45 0.45 1.0', 'role': 'object_to_right_zone'},
}

ZONES = {
    'left_zone': {'pos': (-0.25, 0.12, TABLE_TOP_Z + 0.004), 'size': (0.105, 0.080, 0.004), 'rgba': '0.20 0.75 0.35 0.35'},
    'right_zone': {'pos': (0.25, 0.02, TABLE_TOP_Z + 0.004), 'size': (0.105, 0.080, 0.004), 'rgba': '0.35 0.55 0.95 0.35'},
    'back_zone': {'pos': (0.00, 0.23, TABLE_TOP_Z + 0.004), 'size': (0.105, 0.070, 0.004), 'rgba': '0.95 0.70 0.20 0.35'},
}

SCENARIO_STATES = [
    {
        'scene_id': 'sort_story_00_needs_sorting',
        'scene_status': 'needs_sorting',
        'state_caption': '정리가 필요한 초기 상태: 파란 블록이 빨간 상자 앞을 막고 여러 물체가 흩어져 있다.',
        'transition_from_previous': '시나리오 시작 상태이다.',
        'draft_next_process': '파란 장애물을 오른쪽으로 밀어 빨간 상자로 가는 작업 경로를 만든다.',
        'positions': {'target_red_box': (0.00, 0.06), 'blue_blocker': (0.00, -0.08), 'yellow_cylinder': (-0.10, 0.13), 'gray_box': (0.12, 0.02)},
    },
    {
        'scene_id': 'sort_story_01_after_push_right',
        'scene_status': 'in_progress',
        'state_caption': '파란 블록을 오른쪽으로 밀어 목표 상자 앞 경로가 열렸다.',
        'transition_from_previous': '파란 장애물을 오른쪽으로 밀었다.',
        'draft_next_process': '빨간 목표 상자를 들어서 왼쪽 보관 영역으로 옮긴다.',
        'positions': {'target_red_box': (0.00, 0.06), 'blue_blocker': (0.25, -0.08), 'yellow_cylinder': (-0.10, 0.13), 'gray_box': (0.12, 0.02)},
    },
    {
        'scene_id': 'sort_story_02_after_lift_left',
        'scene_status': 'in_progress',
        'state_caption': '빨간 목표 상자가 왼쪽 보관 영역에 놓였지만 아직 노란 원통과 회색 블록이 남아 있다.',
        'transition_from_previous': '빨간 목표 상자를 들어서 왼쪽 보관 영역으로 옮겼다.',
        'draft_next_process': '노란 원통을 들어서 뒤쪽 정리 영역으로 옮긴다.',
        'positions': {'target_red_box': (-0.25, 0.12), 'blue_blocker': (0.25, -0.08), 'yellow_cylinder': (-0.10, 0.13), 'gray_box': (0.12, 0.02)},
    },
    {
        'scene_id': 'sort_story_03_after_yellow_back',
        'scene_status': 'in_progress',
        'state_caption': '노란 원통이 뒤쪽 정리 영역에 놓였고 회색 블록 하나가 아직 정리되지 않았다.',
        'transition_from_previous': '노란 원통을 들어서 뒤쪽 정리 영역으로 옮겼다.',
        'draft_next_process': '회색 블록을 들어서 오른쪽 정리 영역으로 옮긴다.',
        'positions': {'target_red_box': (-0.25, 0.12), 'blue_blocker': (0.25, -0.08), 'yellow_cylinder': (0.00, 0.23), 'gray_box': (0.12, 0.02)},
    },
    {
        'scene_id': 'sort_story_04_sorted_done',
        'scene_status': 'sorted',
        'state_caption': '정리가 완료된 상태: 각 물체가 지정된 영역에 놓여 있다.',
        'transition_from_previous': '회색 블록을 들어서 오른쪽 정리 영역으로 옮겼다.',
        'draft_next_process': '정리 완료 상태를 확인하고 더 이상 움직이지 않는다.',
        'positions': {'target_red_box': (-0.25, 0.12), 'blue_blocker': (0.25, -0.08), 'yellow_cylinder': (0.00, 0.23), 'gray_box': (0.25, 0.02)},
    },
]

print('states:', len(SCENARIO_STATES))
print(Counter(s['scene_status'] for s in SCENARIO_STATES))
for state in SCENARIO_STATES:
    print('-', state['scene_id'], '|', state['draft_next_process'])

---
## 2. MuJoCo XML 생성과 렌더링

각 상태를 XML로 만들고 이미지로 렌더링합니다. 생성된 XML은 그대로 `mujoco-scene-editor`에서 열어 추가 편집할 수 있습니다.

In [ ]:
# 2-1. MuJoCo XML 생성 함수

def object_z(geom_type, size):
    if geom_type == 'sphere':
        return TABLE_TOP_Z + size[0]
    return TABLE_TOP_Z + size[2]

def size_to_mjcf(geom_type, size):
    sx, sy, sz = size
    if geom_type == 'box':
        return f'{sx:.3f} {sy:.3f} {sz:.3f}'
    if geom_type == 'cylinder':
        return f'{sx:.3f} {sz:.3f}'
    if geom_type == 'sphere':
        return f'{sx:.3f}'
    return f'{sx:.3f} {sy:.3f} {sz:.3f}'

def object_body_xml(name, xy, spec):
    x, y = xy
    geom_type = spec['geom_type']
    z = object_z(geom_type, spec['size'])
    size_str = size_to_mjcf(geom_type, spec['size'])
    rgba = spec['rgba']
    return f'''
    <body name="{name}" pos="{x:.3f} {y:.3f} {z:.3f}">
      <geom name="{name}_geom" type="{geom_type}" size="{size_str}" rgba="{rgba}" mass="0.05" friction="0.8 0.02 0.001"/>
    </body>'''

def zone_geom_xml(name, zone):
    x, y, z = zone['pos']
    sx, sy, sz = zone['size']
    return f'<geom name="{name}" type="box" pos="{x:.3f} {y:.3f} {z:.3f}" size="{sx:.3f} {sy:.3f} {sz:.3f}" rgba="{zone["rgba"]}" contype="0" conaffinity="0"/>'

def make_sorting_scene_xml(state):
    zones = '\n    '.join(zone_geom_xml(name, zone) for name, zone in ZONES.items())
    bodies = '\n'.join(object_body_xml(name, state['positions'][name], OBJECT_SPECS[name]) for name in OBJECT_SPECS)
    return f'''<mujoco model="{state['scene_id']}">
  <compiler angle="degree"/>
  <option gravity="0 0 -9.81" timestep="0.002"/>
  <visual>
    <global offwidth="640" offheight="480"/>
  </visual>
  <worldbody>
    <light name="key_light" pos="0 -1.4 2.6" dir="0 0 -1" diffuse="0.9 0.9 0.9"/>
    <geom name="floor" type="plane" pos="0 0 0" size="1.5 1.5 0.02" rgba="0.50 0.54 0.58 1"/>
    <geom name="table" type="box" pos="0 0 0.390" size="0.46 0.36 0.035" rgba="0.72 0.72 0.68 1"/>
    {zones}
    <camera name="vlm_camera" pos="0 -0.82 0.86" xyaxes="1 0 0 0 0.62 0.78" fovy="48"/>
{bodies}
  </worldbody>
</mujoco>'''

for state in SCENARIO_STATES:
    ET.fromstring(make_sorting_scene_xml(state))
print('✓ XML string parse 통과')

In [ ]:
# 2-2. 렌더링과 저장 함수

def render_xml(xml_text, camera='vlm_camera', width=640, height=480, steps=100):
    if mujoco is None:
        raise RuntimeError('mujoco가 설치되어 있지 않습니다. 0-1 설치 셀을 먼저 실행하세요.')
    model = mujoco.MjModel.from_xml_string(xml_text)
    data = mujoco.MjData(model)
    for _ in range(steps):
        mujoco.mj_step(model, data)
    renderer = mujoco.Renderer(model, width=width, height=height)
    renderer.update_scene(data, camera=camera)
    image = renderer.render().copy()
    renderer.close()
    return image

def show_rgb(image, title=None, figsize=(5, 4)):
    plt.figure(figsize=figsize)
    plt.imshow(image)
    plt.axis('off')
    if title:
        plt.title(title)
    plt.show()

In [ ]:
# 2-3. 정리 시나리오 씬 생성

scene_records = []

for state in SCENARIO_STATES:
    xml_text = make_sorting_scene_xml(state)
    xml_path = XML_DIR / f"{state['scene_id']}.xml"
    image_path = IMAGE_DIR / f"{state['scene_id']}.png"
    xml_path.write_text(xml_text, encoding='utf-8')

    if mujoco is not None:
        image = render_xml(xml_text)
        Image.fromarray(image).save(image_path)
    else:
        image_path = None

    scene_records.append({
        'scene_id': state['scene_id'],
        'scene_status': state['scene_status'],
        'state_caption': state['state_caption'],
        'transition_from_previous': state['transition_from_previous'],
        'draft_next_process': state['draft_next_process'],
        'xml_path': str(xml_path),
        'image_path': str(image_path) if image_path else None,
        'positions': state['positions'],
    })

print('✓ XML 저장:', XML_DIR)
if mujoco is not None:
    print('✓ 이미지 저장:', IMAGE_DIR)
    fig, axes = plt.subplots(1, len(scene_records), figsize=(17, 3.5))
    for ax, record in zip(axes, scene_records):
        img = Image.open(record['image_path'])
        ax.imshow(img)
        ax.axis('off')
        ax.set_title(record['scene_id'].replace('sort_story_', ''), fontsize=8)
    plt.tight_layout()
    plt.show()
else:
    print('⚠️ mujoco가 없어 이미지는 아직 생성하지 않았습니다.')

---
## 3. MuJoCo scene editor로 편집하기

자동 생성된 XML은 기본 시나리오입니다. 더 좋은 카메라 구도나 물체 배치를 만들고 싶으면 `mujoco-scene-editor`로 XML을 열어 편집합니다.

```bash
mjedit /content/session10_process_dataset/data/xml/sort_story_00_needs_sorting.xml
```

편집 후 Export한 XML을 `edited_xml/` 폴더에 넣고 아래 셀로 다시 렌더링합니다. Colab 포트 접근이 어렵다면 로컬 PC에서 편집한 뒤 Google Drive에 업로드해도 됩니다.

In [ ]:
# 3-1. 편집 명령 안내

for record in scene_records:
    print(f'mjedit "{record["xml_path"]}"')

RUN_MJEDIT_IN_NOTEBOOK = False

if RUN_MJEDIT_IN_NOTEBOOK:
    import subprocess
    first_xml = scene_records[0]['xml_path']
    proc = subprocess.Popen(['mjedit', first_xml, '--skip-open-browser'])
    print('mjedit process id:', proc.pid)
    print('출력되는 Local URL을 Colab 포트 프록시 또는 브라우저로 열어 확인하세요.')
else:
    print('RUN_MJEDIT_IN_NOTEBOOK=False: 명령만 출력합니다.')

In [ ]:
# 3-2. 편집된 XML이 있으면 다시 렌더링하고 scene_records에 반영

edited_xml_files = sorted(EDITED_XML_DIR.glob('*.xml'))
scene_by_id = {record['scene_id']: record for record in scene_records}

if not edited_xml_files:
    print('edited_xml 폴더에 XML이 없습니다. 자동 생성 XML을 그대로 사용합니다.')
else:
    for xml_path in edited_xml_files:
        xml_text = xml_path.read_text(encoding='utf-8')
        ET.fromstring(xml_text)
        image_path = IMAGE_DIR / f'{xml_path.stem}.png'

        if mujoco is not None:
            image = render_xml(xml_text)
            Image.fromarray(image).save(image_path)
            show_rgb(image, f'edited: {xml_path.name}')
            print('saved:', image_path)
        else:
            image_path = None
            print('XML parse ok, 렌더링은 Colab/MuJoCo 환경에서 실행하세요:', xml_path)

        # 파일명이 기존 scene_id와 같으면 해당 record를 편집본으로 교체합니다.
        # 예: edited_xml/sort_story_00_needs_sorting.xml
        if xml_path.stem in scene_by_id:
            record = scene_by_id[xml_path.stem]
            record['xml_path'] = str(xml_path)
            if image_path is not None:
                record['image_path'] = str(image_path)
            record['edited_with_mujoco_scene_editor'] = True
            print('updated record:', xml_path.stem)
        else:
            print('참고: 파일명이 기존 scene_id와 달라 자동 라벨에는 포함하지 않았습니다:', xml_path.name)


---
## 4. 과정 설명을 먼저 만들고, 그 다음 분류한다

1. 각 씬을 보고 “다음에 무엇을 해야 하는가?”를 과정 설명으로 적는다.
2. 설명들을 모아 본 뒤, 이 시나리오에 맞는 라벨 체계를 만든다.
3. 필요한 경우에만 예전의 `pick/push/reorder/ignore`와 연결한다.

즉, 라벨이 먼저가 아니라 **씬과 과정 설명이 먼저**입니다.

In [ ]:
# 4-1. 초안 과정 설명 확인

for i, record in enumerate(scene_records):
    print(f'[{i}] {record["scene_id"]}')
    print('상태:', record['state_caption'])
    print('이전 과정:', record['transition_from_previous'])
    print('다음 과정 초안:', record['draft_next_process'])
    print()

In [ ]:
# 4-2. 과정 설명을 보고 시나리오 전용 라벨 체계 확정

PROCESS_TAXONOMY = {
    'slide_obstacle_right': {'process_type': 'planar_push', 'short_ko': '오른쪽으로 밀기', 'definition': '장애물을 집지 않고 테이블 위에서 오른쪽으로 밀어 경로를 확보한다.', 'legacy_action_optional': 'push'},
    'lift_target_to_left_zone': {'process_type': 'pick_and_place', 'short_ko': '들어서 왼쪽으로 옮기기', 'definition': '목표 물체를 들어 왼쪽 보관 영역에 놓는다.', 'legacy_action_optional': 'pick'},
    'lift_object_to_back_zone': {'process_type': 'pick_and_place', 'short_ko': '들어서 뒤쪽으로 옮기기', 'definition': '남은 물체를 들어 뒤쪽 정리 영역에 놓는다.', 'legacy_action_optional': 'reorder'},
    'lift_object_to_right_zone': {'process_type': 'pick_and_place', 'short_ko': '들어서 오른쪽으로 옮기기', 'definition': '남은 물체를 들어 오른쪽 정리 영역에 놓는다.', 'legacy_action_optional': 'reorder'},
    'confirm_sorted_stop': {'process_type': 'terminal_check', 'short_ko': '정리 완료 확인', 'definition': '정리가 끝났는지 확인하고 추가 조작 없이 멈춘다.', 'legacy_action_optional': 'ignore'},
}

PROCESS_ASSIGNMENTS = {
    'sort_story_00_needs_sorting': 'slide_obstacle_right',
    'sort_story_01_after_push_right': 'lift_target_to_left_zone',
    'sort_story_02_after_lift_left': 'lift_object_to_back_zone',
    'sort_story_03_after_yellow_back': 'lift_object_to_right_zone',
    'sort_story_04_sorted_done': 'confirm_sorted_stop',
}

for label, info in PROCESS_TAXONOMY.items():
    print(label, '|', info['short_ko'], '|', info['process_type'], '| legacy:', info['legacy_action_optional'])

In [ ]:
# 4-3. 최종 라벨과 상세 설명 확정

FINAL_DETAIL_OVERRIDES = {
    'sort_story_00_needs_sorting': '파란 블록이 빨간 목표 상자 앞쪽 경로를 막고 있으므로, 먼저 오른쪽으로 밀어 접근 공간을 만든다.',
    'sort_story_01_after_push_right': '장애물이 오른쪽으로 이동해 빨간 목표 상자에 접근할 수 있으므로, 목표 상자를 들어 왼쪽 보관 영역에 놓는다.',
    'sort_story_02_after_lift_left': '빨간 목표 상자는 정리되었지만 노란 원통이 아직 작업 영역에 있으므로, 뒤쪽 정리 영역으로 옮긴다.',
    'sort_story_03_after_yellow_back': '노란 원통까지 정리되었고 회색 블록만 남아 있으므로, 회색 블록을 오른쪽 정리 영역으로 옮긴다.',
    'sort_story_04_sorted_done': '빨간 상자, 노란 원통, 회색 블록이 모두 지정된 영역에 있으므로 추가 조작 없이 완료 상태를 확인한다.',
}

SFT_INSTRUCTION = (
    '이미지를 보고 로봇이 다음에 수행해야 할 정리 과정을 판단하세요. '
    '처음부터 pick/push/reorder/ignore로 답하지 말고, 이 시나리오의 과정 라벨을 사용하세요. '
    '응답은 반드시 JSON 하나만 출력하세요.'
)

final_records = []

for record in scene_records:
    process_label = PROCESS_ASSIGNMENTS[record['scene_id']]
    taxonomy = PROCESS_TAXONOMY[process_label]
    answer = {
        'process_label': process_label,
        'process_type': taxonomy['process_type'],
        'process_short': taxonomy['short_ko'],
        'next_process_description': record['draft_next_process'],
        'scene_status': record['scene_status'],
        'reason': FINAL_DETAIL_OVERRIDES[record['scene_id']],
    }
    final_records.append({
        **record,
        'instruction': SFT_INSTRUCTION,
        'final_label': {**answer, 'legacy_action_optional': taxonomy['legacy_action_optional']},
        'assistant_answer': json.dumps(answer, ensure_ascii=False),
    })

print('final records:', len(final_records))
print(Counter(r['final_label']['process_label'] for r in final_records))
print(json.dumps(final_records[0]['final_label'], ensure_ascii=False, indent=2))

---
## 5. 파인튜닝용 데이터셋 만들기

SmolVLM SFT 데이터셋은 이미지와 대화 메시지의 쌍으로 만듭니다. assistant 답변은 최종 라벨 JSON입니다.

In [ ]:
# 5-1. 라벨 파일, manifest, SFT JSONL 저장

scenario_manifest_path = LABEL_DIR / 'scenario_process_manifest.jsonl'
sft_jsonl_path = DATASET_DIR / 'smolvlm_sft_records.jsonl'
label_json_path = LABEL_DIR / 'final_labels.json'

label_json_path.write_text(json.dumps(final_records, ensure_ascii=False, indent=2), encoding='utf-8')

with scenario_manifest_path.open('w', encoding='utf-8') as f_manifest, sft_jsonl_path.open('w', encoding='utf-8') as f_sft:
    for record in final_records:
        f_manifest.write(json.dumps(record, ensure_ascii=False) + '\n')
        sft_row = {
            'scene_id': record['scene_id'],
            'image_path': record['image_path'],
            'xml_path': record['xml_path'],
            'target_process_label': record['final_label']['process_label'],
            'target_process_type': record['final_label']['process_type'],
            'messages': [
                {'role': 'user', 'content': [{'type': 'image'}, {'type': 'text', 'text': record['instruction']}]},
                {'role': 'assistant', 'content': [{'type': 'text', 'text': record['assistant_answer']}]},
            ],
        }
        f_sft.write(json.dumps(sft_row, ensure_ascii=False) + '\n')

print('label_json:', label_json_path)
print('manifest  :', scenario_manifest_path)
print('sft_jsonl :', sft_jsonl_path)
print('rows:', len(final_records))

In [ ]:
# 5-2. Hugging Face Dataset 객체로 변환

try:
    from datasets import Dataset
    DATASETS_AVAILABLE = True
except Exception as e:
    DATASETS_AVAILABLE = False
    print('datasets import 실패:', repr(e))

def to_sft_row(record):
    return {
        'scene_id': record['scene_id'],
        'image_path': record['image_path'],
        'xml_path': record['xml_path'],
        'scene_status': record['scene_status'],
        'target_process_label': record['final_label']['process_label'],
        'target_process_type': record['final_label']['process_type'],
        'assistant_answer': record['assistant_answer'],
        'messages': [
            {'role': 'user', 'content': [{'type': 'image'}, {'type': 'text', 'text': record['instruction']}]},
            {'role': 'assistant', 'content': [{'type': 'text', 'text': record['assistant_answer']}]},
        ],
    }

def attach_image(example):
    example = dict(example)
    if example['image_path'] and Path(example['image_path']).exists():
        example['images'] = [Image.open(example['image_path']).convert('RGB')]
    else:
        example['images'] = []
    return example

sft_rows = [to_sft_row(record) for record in final_records]

if DATASETS_AVAILABLE:
    raw_dataset = Dataset.from_list(sft_rows)
    sft_dataset = raw_dataset.with_transform(attach_image)
    print(raw_dataset)
    if len(sft_dataset):
        sample = sft_dataset[0]
        print('sample scene:', sample['scene_id'])
        print('target:', sample['target_process_label'])
        print(json.dumps(sample['messages'], ensure_ascii=False, indent=2))
        if sample['images']:
            display(sample['images'][0].resize((426, 320)))
else:
    sft_dataset = None
    print('datasets 패키지가 없어 JSONL만 저장했습니다.')

In [ ]:
# 5-3. 데이터셋 검수

required_record_keys = {'scene_id', 'scene_status', 'state_caption', 'draft_next_process', 'final_label', 'assistant_answer'}
required_answer_keys = {'process_label', 'process_type', 'process_short', 'next_process_description', 'scene_status', 'reason'}

errors = []
for record in final_records:
    missing = required_record_keys - set(record)
    answer = json.loads(record['assistant_answer'])
    missing_answer = required_answer_keys - set(answer)
    image_ok = bool(record['image_path']) and Path(record['image_path']).exists()
    xml_ok = Path(record['xml_path']).exists()
    if missing or missing_answer or not xml_ok or (mujoco is not None and not image_ok):
        errors.append({'scene_id': record['scene_id'], 'missing': sorted(missing), 'missing_answer': sorted(missing_answer), 'image_ok': image_ok, 'xml_ok': xml_ok})

print('records:', len(final_records))
print('status:', Counter(r['scene_status'] for r in final_records))
print('process labels:', Counter(r['final_label']['process_label'] for r in final_records))

if errors:
    print('⚠️ 검수 오류')
    print(json.dumps(errors, ensure_ascii=False, indent=2))
else:
    print('✓ 데이터셋 검수 통과')

---
## 6. Google Drive 팀 폴더에 저장

이전 세션과 마찬가지로 팀 공유 폴더에 결과를 저장합니다. 저장 대상은 XML, 이미지, 최종 라벨, SFT JSONL입니다.

In [ ]:
# 6-1. Drive 저장 블록

DRIVE_SAVE_NAME = 'session10_process_sft_dataset'

if TEAM_ROOT and TEAM_ROOT.exists():
    drive_dataset_dir = TEAM_ROOT / 'data' / DRIVE_SAVE_NAME
    drive_dataset_dir.mkdir(parents=True, exist_ok=True)

    for subdir in ['xml', 'images', 'labels', 'sft_dataset']:
        src = DATA_DIR / subdir
        dst = drive_dataset_dir / subdir
        if src.exists():
            shutil.copytree(src, dst, dirs_exist_ok=True)
            print('copied:', src, '->', dst)

    summary = {
        'dataset_name': DRIVE_SAVE_NAME,
        'local_data_dir': str(DATA_DIR),
        'drive_dataset_dir': str(drive_dataset_dir),
        'num_records': len(final_records),
        'scene_status_counts': dict(Counter(r['scene_status'] for r in final_records)),
        'process_label_counts': dict(Counter(r['final_label']['process_label'] for r in final_records)),
        'files': {
            'manifest': str(drive_dataset_dir / 'labels' / 'scenario_process_manifest.jsonl'),
            'sft_jsonl': str(drive_dataset_dir / 'sft_dataset' / 'smolvlm_sft_records.jsonl'),
        },
    }
    summary_path = drive_dataset_dir / 'dataset_summary.json'
    summary_path.write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding='utf-8')
    print('✓ Drive 저장 완료:', drive_dataset_dir)
    print('summary:', summary_path)
else:
    print('TEAM_ROOT가 없어서 Drive 저장을 건너뜁니다.')
    print('로컬 결과 폴더:', DATA_DIR)

---
## 7. 선택: SmolVLM SFTTrainer에 넣기 전 형태 확인

아래 형태가 바로 TRL `SFTTrainer`에 전달할 기본 구조입니다. 학습 셀을 붙일 때는 `images`가 실제 PIL 이미지로 로드되는지, assistant 답변이 JSON 문자열인지 확인하세요.

In [ ]:
# 7-1. SFTTrainer 입력 샘플 확인

if DATASETS_AVAILABLE and sft_dataset is not None and len(sft_dataset):
    ex = sft_dataset[0]
    print('scene_id:', ex['scene_id'])
    print('target_process_label:', ex['target_process_label'])
    print('has image:', bool(ex['images']))
    print('assistant answer:')
    print(ex['assistant_answer'])
else:
    print('Dataset 객체가 없으면 smolvlm_sft_records.jsonl을 먼저 확인하세요:', sft_jsonl_path)

---
## 8. 미니 과제

1. `sort_story_00_needs_sorting.xml`을 `mjedit`로 열어 물체 배치를 조금 바꿉니다.
2. Export한 XML을 `edited_xml/`에 넣고 다시 렌더링합니다.
3. 그 장면에서 자연스러운 “다음 과정 설명”을 먼저 씁니다.
4. 기존 `PROCESS_TAXONOMY`에 맞으면 해당 라벨을 붙이고, 맞지 않으면 새 라벨을 추가합니다.
5. `scenario_process_manifest.jsonl`과 `smolvlm_sft_records.jsonl`을 다시 만든 뒤 Drive에 저장합니다.

체크포인트:

* 정리가 필요한 상태와 완료된 상태가 모두 들어 있는가?
* 중간 과정 씬마다 설명이 붙어 있는가?
* 라벨은 장면을 본 뒤 정했는가?
* assistant 답변은 JSON 하나로 파싱 가능한가?
* Google Drive 팀 폴더에 XML, 이미지, 라벨, SFT JSONL이 모두 저장되었는가?